# 1. Estrutura da tabela

In [0]:
%sql
DESCRIBE dev_procurement.corp_curated.tbl_ds_mdm_zglmm159;


# 2. Metadados da tabela

In [0]:
%sql
DESCRIBE DETAIL dev_procurement.corp_curated.tbl_ds_mdm_zglmm159;


# 3. Histórico das cargas

In [0]:
%sql
DESCRIBE HISTORY dev_procurement.corp_curated.tbl_ds_mdm_zglmm159;

# 4. Quantidade de registros

In [0]:
%sql
SELECT COUNT(*) AS qtd_registros
FROM dev_procurement.corp_curated.tbl_ds_mdm_zglmm159;

# 5. Amostra dos dados

In [0]:
%sql
SELECT *
FROM dev_procurement.corp_curated.tbl_ds_mdm_zglmm159
LIMIT 20;

# 6. Quantidade de colunas

In [0]:
%sql
SELECT COUNT(*) AS qtd_colunas
FROM dev_procurement.information_schema.columns
WHERE table_schema = 'corp_curated'
  AND table_name = 'tbl_ds_mdm_zglmm159';

# 7. Quantidade de colunas

In [0]:
%sql
SELECT COUNT(*) AS qtd_colunas
FROM dev_procurement.information_schema.columns
WHERE table_schema = 'corp_curated'
  AND table_name = 'tbl_ds_mdm_zglmm159'
  AND column_name NOT LIKE '#%';

# 8. Validar chave Material + Centro

In [0]:
%sql
SELECT
  COUNT(*) AS total,
  COUNT(DISTINCT CONCAT(cod_material,'|',cod_centro)) AS distintos
FROM dev_procurement.corp_curated.tbl_ds_mdm_zglmm159;

# 9. Quantidade de Materiais

In [0]:
%sql
SELECT COUNT(DISTINCT cod_material) AS qtd_materiais
FROM dev_procurement.corp_curated.tbl_ds_mdm_zglmm159;

# 10. Quantidade de Centros

In [0]:
%sql
SELECT COUNT(DISTINCT cod_centro) AS qtd_centros
FROM dev_procurement.corp_curated.tbl_ds_mdm_zglmm159;

# 11. Verificar duplicidades

In [0]:
%sql
SELECT
  cod_material,
  cod_centro,
  COUNT(*) AS qtd
FROM dev_procurement.corp_curated.tbl_ds_mdm_zglmm159
GROUP BY
  cod_material,
  cod_centro
HAVING COUNT(*) > 1
LIMIT 100;

# 12. Unicidade da chave

In [0]:
%sql
SELECT
  COUNT(*) AS qtd_total,
  COUNT(DISTINCT CONCAT(cod_material,'|',cod_centro)) AS qtd_chaves
FROM dev_procurement.corp_curated.tbl_ds_mdm_zglmm159;

# 13. Campos obrigatórios preenchidos

In [0]:
%sql
SELECT
  COUNT(*) AS total,
  SUM(CASE WHEN cod_material IS NULL THEN 1 ELSE 0 END) AS material_nulo,
  SUM(CASE WHEN cod_centro IS NULL THEN 1 ELSE 0 END) AS centro_nulo
FROM dev_procurement.corp_curated.tbl_ds_mdm_zglmm159;

# 14. Distribuição por centro

In [0]:
%sql
SELECT
  cod_centro,
  COUNT(*) AS qtd
FROM dev_procurement.corp_curated.tbl_ds_mdm_zglmm159
GROUP BY cod_centro
ORDER BY qtd DESC;

# 15. Perfil dos tipos MRP

In [0]:
%sql
SELECT
  tp_mrp,
  COUNT(*) AS qtd
FROM dev_procurement.corp_curated.tbl_ds_mdm_zglmm159
GROUP BY tp_mrp
ORDER BY qtd DESC;

# 16. Detalhar chaves duplicadas

In [0]:
%sql
-- Lista as chaves material+centro que aparecem mais de uma vez (esperado: 1 linha por chave)
SELECT
  cod_material,
  cod_centro,
  COUNT(*) AS qtd_repeticoes
FROM dev_procurement.corp_curated.tbl_ds_mdm_zglmm159
GROUP BY cod_material, cod_centro
HAVING COUNT(*) > 1
ORDER BY qtd_repeticoes DESC, cod_material, cod_centro;

# 17. Linhas completas das chaves duplicadas

In [0]:
%sql
-- Mostra as linhas COMPLETAS das chaves duplicadas, para ver o que difere entre elas
WITH duplicadas AS (
  SELECT cod_material, cod_centro
  FROM dev_procurement.corp_curated.tbl_ds_mdm_zglmm159
  GROUP BY cod_material, cod_centro
  HAVING COUNT(*) > 1
)
SELECT t.*
FROM dev_procurement.corp_curated.tbl_ds_mdm_zglmm159 t
INNER JOIN duplicadas d
  ON t.cod_material = d.cod_material
  AND t.cod_centro = d.cod_centro
ORDER BY t.cod_material, t.cod_centro;

# 18. % de nulos por coluna crítica

In [0]:
%sql
-- Percentual de nulos das colunas mais importantes para o teste
SELECT
  COUNT(*) AS total_linhas,
  ROUND(100.0 * SUM(CASE WHEN cod_empresa IS NULL THEN 1 ELSE 0 END) / COUNT(*), 2) AS pct_null_empresa,
  ROUND(100.0 * SUM(CASE WHEN tp_avaliacao IS NULL THEN 1 ELSE 0 END) / COUNT(*), 2) AS pct_null_tp_avaliacao,
  ROUND(100.0 * SUM(CASE WHEN cod_classe_avaliacao IS NULL THEN 1 ELSE 0 END) / COUNT(*), 2) AS pct_null_classe_aval,
  ROUND(100.0 * SUM(CASE WHEN vl_preco_medio_movel IS NULL THEN 1 ELSE 0 END) / COUNT(*), 2) AS pct_null_preco_medio,
  ROUND(100.0 * SUM(CASE WHEN tp_mrp IS NULL THEN 1 ELSE 0 END) / COUNT(*), 2) AS pct_null_tp_mrp,
  ROUND(100.0 * SUM(CASE WHEN cod_abc IS NULL THEN 1 ELSE 0 END) / COUNT(*), 2) AS pct_null_cod_abc,
  ROUND(100.0 * SUM(CASE WHEN cod_status_material_centro IS NULL THEN 1 ELSE 0 END) / COUNT(*), 2) AS pct_null_status
FROM dev_procurement.corp_curated.tbl_ds_mdm_zglmm159;

# 19. Contagem por empresa

In [0]:
%sql
-- Volume por empresa (ajuda a bater o total contra o SAP por ledger/empresa)
SELECT
  cod_empresa,
  COUNT(*) AS qtd_linhas,
  COUNT(DISTINCT cod_material) AS qtd_materiais,
  COUNT(DISTINCT cod_centro) AS qtd_centros
FROM dev_procurement.corp_curated.tbl_ds_mdm_zglmm159
GROUP BY cod_empresa
ORDER BY qtd_linhas DESC;

# 20. Janela temporal da base

In [0]:
%sql
-- Menor e maior data de início de validade do status -> confirma o "corte" da base
SELECT
  MIN(dt_inicio_validade_status) AS data_mais_antiga,
  MAX(dt_inicio_validade_status) AS data_mais_recente,
  COUNT(DISTINCT dt_inicio_validade_status) AS qtd_datas_distintas
FROM dev_procurement.corp_curated.tbl_ds_mdm_zglmm159;

# 21. Distribuição tipo de avaliação

In [0]:
%sql
-- Distribuição de tipo de avaliação
SELECT
  tp_avaliacao,
  COUNT(*) AS qtd
FROM dev_procurement.corp_curated.tbl_ds_mdm_zglmm159
GROUP BY tp_avaliacao
ORDER BY qtd DESC;

# 22. Distribuição classe de avaliação

In [0]:
%sql
-- Distribuição de classe de avaliação (conta contábil)
SELECT
  cod_classe_avaliacao,
  COUNT(*) AS qtd
FROM dev_procurement.corp_curated.tbl_ds_mdm_zglmm159
GROUP BY cod_classe_avaliacao
ORDER BY qtd DESC;

# 23. Validar chave REAL (material+centro+tp_avaliacao)

In [0]:
%sql
-- 18. Validar a chave REAL (material+centro+tp_avaliacao) -> deve dar 0 duplicadas
SELECT COUNT(*) AS chaves_repetidas
FROM (
  SELECT cod_material, cod_centro, tp_avaliacao, COUNT(*) AS qtd
  FROM dev_procurement.corp_curated.tbl_ds_mdm_zglmm159
  GROUP BY cod_material, cod_centro, tp_avaliacao
  HAVING COUNT(*) > 1
);

# 24. Dimensionar split valuation por tp_avaliacao

In [0]:
%sql
-- 19. Dimensionar o split valuation: quantos materiais/centros têm mais de um tp_avaliacao
SELECT tp_avaliacao, COUNT(*) AS qtd_linhas
FROM dev_procurement.corp_curated.tbl_ds_mdm_zglmm159
GROUP BY tp_avaliacao
ORDER BY qtd_linhas DESC;

# 25. Checar inconsistência de zeros à esquerda no tp_avaliacao

In [0]:
%sql
-- 20. Checar a inconsistência de zeros à esquerda no tp_avaliacao
SELECT
  tp_avaliacao,
  LENGTH(tp_avaliacao) AS tamanho,
  COUNT(*) AS qtd
FROM dev_procurement.corp_curated.tbl_ds_mdm_zglmm159
WHERE tp_avaliacao RLIKE '^[0-9]+$' -- só os numéricos
GROUP BY tp_avaliacao, LENGTH(tp_avaliacao)
ORDER BY tp_avaliacao;

# 26. Escolha dos centros para analise

In [0]:
%sql
SELECT
  cod_empresa,
  cod_centro,
  COUNT(*)                                                          AS linhas_esperadas_sap,
  COUNT(DISTINCT cod_material)                                      AS materiais,
  SUM(CASE WHEN NULLIF(TRIM(tp_avaliacao),'') IS NOT NULL THEN 1 ELSE 0 END) AS linhas_split_valuation,
  SUM(CASE WHEN tp_avaliacao RLIKE '^[0-9]+$' THEN 1 ELSE 0 END)    AS registros_suspeitos,
  ROUND(100.0 * SUM(CASE WHEN tp_mrp IS NULL THEN 1 ELSE 0 END)/COUNT(*),1) AS pct_sem_mrp,
  ROUND(100.0 * SUM(CASE WHEN cod_abc IS NULL THEN 1 ELSE 0 END)/COUNT(*),1) AS pct_sem_abc
FROM dev_procurement.corp_curated.tbl_ds_mdm_zglmm159
WHERE cod_empresa IN ('4014','4008')
GROUP BY cod_empresa, cod_centro
ORDER BY cod_empresa, linhas_esperadas_sap DESC;

In [0]:
%sql
DESCRIBE TABLE dev_procurement.corp_curated.tbl_ds_mdm_zglmm159;

In [0]:
%sql
